In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parent))

In [2]:
import numpy as np
import pandas as pd

from src.data.loader import load_data
from src.data.splitter import stratified_train_test_split

from src.preprocessing.cleaning import remove_duplicates
from src.preprocessing.label_encoding import LabelEncoder
from src.preprocessing.scaling import StandardScaler

In [3]:
df_raw = load_data()
print(f"Kích thước dữ liệu gốc: {df_raw.shape}")
df_raw.head()

Kích thước dữ liệu gốc: (13611, 17)


,Area,Perimeter,MajorAxisLength,MinorAxisLength,AspectRation,Eccentricity,ConvexArea,EquivDiameter,Extent,Solidity,roundness,Compactness,ShapeFactor1,ShapeFactor2,ShapeFactor3,ShapeFactor4,Class
0,28395,610.291,208.178117,173.888747,1.197191,0.549812,28715,190.141097,0.763923,0.988856,0.958027,0.913358,0.007332,0.003147,0.834222,0.998724,SEKER
1,28734,638.018,200.524796,182.734419,1.097356,0.411785,29172,191.272750,0.783968,0.984986,0.887034,0.953861,0.006979,0.003564,0.909851,0.998430,SEKER
2,29380,624.110,212.826130,175.931143,1.209713,0.562727,29690,193.410904,0.778113,0.989559,0.947849,0.908774,0.007244,0.003048,0.825871,0.999066,SEKER
3,30008,645.884,210.557999,182.516516,1.153638,0.498616,30724,195.467062,0.782681,0.976696,0.903936,0.928329,0.007017,0.003215,0.861794,0.994199,SEKER
4,30140,620.134,201.847882,190.279279,1.060798,0.333680,30417,195.896503,0.773098,0.990893,0.984877,0.970516,0.006697,0.003665,0.941900,0.999166,SEKER


In [4]:
df_clean = remove_duplicates(df_raw)

num_duplicates = len(df_raw) - len(df_clean)
print(f"Số dòng trùng lặp đã loại bỏ: {num_duplicates}")
print(f"Kích thước dữ liệu sau khi làm sạch: {df_clean.shape}")

Số dòng trùng lặp đã loại bỏ: 68
Kích thước dữ liệu sau khi làm sạch: (13543, 17)


In [5]:
feature_names = df_clean.drop(columns=["Class"]).columns.tolist()

X = df_clean.drop(columns=["Class"]).to_numpy()
y_raw = df_clean["Class"]

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_raw)

label_mapping = dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))
print("Ánh xạ nhãn (Class Mapping):")
for name, idx in label_mapping.items():
    print(f"  {idx}: {name}")

Ánh xạ nhãn (Class Mapping):
  0: BARBUNYA
  1: BOMBAY
  2: CALI
  3: DERMASON
  4: HOROZ
  5: SEKER
  6: SIRA


In [6]:
X_train, X_test, y_train, y_test = stratified_train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Tập huấn luyện (Train): X = {X_train.shape}, y = {y_train.shape}")
print(f"Tập kiểm thử   (Test) : X = {X_test.shape},  y = {y_test.shape}")

Tập huấn luyện (Train): X = (10836, 16), y = (10836,)
Tập kiểm thử   (Test) : X = (2707, 16),  y = (2707,)


In [7]:
print(f"{'Class':<12} {'Train Count':<14} {'Train %':<10} {'Test Count':<14} {'Test %':<10}")
print("-" * 60)
for cls_idx, cls_name in enumerate(label_encoder.classes_):
    n_tr = np.sum(y_train == cls_idx)
    n_te = np.sum(y_test == cls_idx)
    pct_tr = n_tr / len(y_train) * 100
    pct_te = n_te / len(y_test) * 100
    print(f"{cls_name:<12} {n_tr:<14} {pct_tr:.2f}%     {n_te:<14} {pct_te:.2f}%")

Class        Train Count    Train %    Test Count     Test %    
------------------------------------------------------------
BARBUNYA     1058           9.76%     264            9.75%
BOMBAY       418            3.86%     104            3.84%
CALI         1304           12.03%     326            12.04%
DERMASON     2837           26.18%     709            26.19%
HOROZ        1488           13.73%     372            13.74%
SEKER        1622           14.97%     405            14.96%
SIRA         2109           19.46%     527            19.47%


In [8]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Kiểm tra phân phối sau chuẩn hóa:")
print(f"Mean tập Train (kỳ vọng ~ 0): {np.allclose(X_train_scaled.mean(axis=0), 0, atol=1e-7)}")
print(f"Std tập Train  (kỳ vọng = 1): {np.allclose(X_train_scaled.std(axis=0), 1, atol=1e-7)}")

Kiểm tra phân phối sau chuẩn hóa:
Mean tập Train (kỳ vọng ~ 0): True
Std tập Train  (kỳ vọng = 1): True


In [9]:
processed_dir = Path("../data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

output_file = processed_dir / "dry_bean_processed.npz"
np.savez_compressed(
    output_file,
    X_train=X_train_scaled,
    X_test=X_test_scaled,
    y_train=y_train,
    y_test=y_test,
    features=np.array(feature_names),
    classes=label_encoder.classes_
)

print(f"Đã lưu dữ liệu tiền xử lý thành công vào: {output_file.resolve()}")

Đã lưu dữ liệu tiền xử lý thành công vào: D:\Machine_Learning\dry-bean-ml\data\processed\dry_bean_processed.npz


In [10]:
data = np.load(output_file)
print("Các mảng dữ liệu có sẵn để train model:", data.files)
print("X_train shape:", data["X_train"].shape)
print("X_test shape: ", data["X_test"].shape)
print("y_train shape:", data["y_train"].shape)
print("y_test shape: ", data["y_test"].shape)

Các mảng dữ liệu có sẵn để train model: ['X_train', 'X_test', 'y_train', 'y_test', 'features', 'classes']
X_train shape: (10836, 16)
X_test shape:  (2707, 16)
y_train shape: (10836,)
y_test shape:  (2707,)
